# Quantum Finance with nQPU

This notebook demonstrates quantum-enhanced financial computation using the **nQPU SDK**.
We cover four major areas of quantum finance:

1. **Option Pricing** -- Quantum Amplitude Estimation (QAE) vs Black-Scholes analytical solutions
2. **Portfolio Optimization** -- QAOA-based Markowitz optimization vs classical brute-force
3. **Risk Analysis** -- Quantum VaR and CVaR estimation under different distribution assumptions
4. **Trading Signals** -- Quantum regime detection, momentum, and backtesting

All computations use simulated quantum circuits (no hardware required). Data is synthetic.

In [ ]:
import numpy as np
import time

# Inline plotting (use %matplotlib inline in Jupyter)
try:
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    %matplotlib inline
    plt.rcParams["figure.figsize"] = (14, 5)
    plt.rcParams["figure.dpi"] = 120
    HAS_MPL = True
    print("matplotlib loaded -- figures will be displayed inline")
except ImportError:
    HAS_MPL = False
    print("matplotlib not available -- skipping figures")

## 1. Option Pricing: QAE vs Black-Scholes

Quantum Amplitude Estimation (QAE) encodes the asset-price distribution into a quantum state
and estimates the expected discounted payoff with a quadratic speedup over classical Monte Carlo.

We compare QAE option prices against the closed-form Black-Scholes formula for European options,
and also price exotic options (Asian, Barrier) where no closed-form solution exists.

In [ ]:
from nqpu.finance import (
    QuantumOptionPricer,
    OptionType,
    QAEMethod,
    black_scholes_call,
    black_scholes_put,
    black_scholes_delta,
)

# Basic ATM European call
pricer = QuantumOptionPricer(
    spot=100, strike=100, rate=0.05, volatility=0.2, maturity=1.0,
)
result = pricer.price()
bs_call = black_scholes_call(100, 100, 0.05, 0.2, 1.0)

print(f"European Call (ATM, S=K=100):")
print(f"  QAE price:     {result.price:.4f}")
print(f"  Black-Scholes: {bs_call:.4f}")
print(f"  QAE delta:     {result.delta:.4f}")
print(f"  CI:            [{result.confidence_interval[0]:.4f}, {result.confidence_interval[1]:.4f}]")
print(f"  Oracle calls:  {result.num_oracle_calls}")

In [ ]:
# Strike sweep: compare QAE vs BS across strikes
strikes = list(range(80, 125, 5))
qae_prices = []
bs_prices = []

print(f"{'Strike':>8} {'QAE':>10} {'BS':>10} {'Diff':>10}")
print(f"{'-'*8} {'-'*10} {'-'*10} {'-'*10}")

for k in strikes:
    p = QuantumOptionPricer(
        spot=100, strike=k, rate=0.05, volatility=0.2, maturity=1.0,
        _compute_delta=False,
    )
    qae_p = p.price().price
    bs_p = black_scholes_call(100, k, 0.05, 0.2, 1.0)
    qae_prices.append(qae_p)
    bs_prices.append(bs_p)
    print(f"{k:>8} {qae_p:>10.4f} {bs_p:>10.4f} {qae_p - bs_p:>+10.4f}")

In [ ]:
# Exotic options: Asian and Barrier
asian_pricer = QuantumOptionPricer(
    spot=100, strike=100, rate=0.05, volatility=0.2, maturity=1.0,
    option_type=OptionType.ASIAN_CALL,
    num_paths=5000, _compute_delta=False,
)
asian_result = asian_pricer.price()

barrier_pricer = QuantumOptionPricer(
    spot=100, strike=100, rate=0.05, volatility=0.2, maturity=1.0,
    option_type=OptionType.BARRIER_UP_AND_OUT, barrier=120,
    num_paths=5000, _compute_delta=False,
)
barrier_result = barrier_pricer.price()

print(f"Exotic Option Pricing:")
print(f"  Asian Call (QAE):      {asian_result.price:.4f}  (MC ref: {asian_result.analytical_price:.4f})")
print(f"  Barrier Up&Out (QAE):  {barrier_result.price:.4f}  (MC ref: {barrier_result.analytical_price:.4f})")

In [ ]:
# Delta surface: vary spot and strike
spot_range = np.arange(80, 125, 5)
strike_range = np.arange(85, 120, 5)
delta_surface = np.zeros((len(spot_range), len(strike_range)))

for i, s in enumerate(spot_range):
    for j, k in enumerate(strike_range):
        delta_surface[i, j] = black_scholes_delta(
            float(s), float(k), 0.05, 0.2, 1.0, is_call=True
        )

print(f"Delta surface computed: {delta_surface.shape}")
print(f"Delta range: [{delta_surface.min():.4f}, {delta_surface.max():.4f}]")

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Strike sweep
    ax = axes[0]
    ax.plot(strikes, qae_prices, "o-", label="QAE", color="#2196F3")
    ax.plot(strikes, bs_prices, "s--", label="Black-Scholes", color="#FF5722")
    ax.set_xlabel("Strike Price")
    ax.set_ylabel("Option Price")
    ax.set_title("European Call: QAE vs Black-Scholes")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Pricing error
    ax = axes[1]
    errors = [q - b for q, b in zip(qae_prices, bs_prices)]
    ax.bar(strikes, errors, width=3, color="#4CAF50", alpha=0.7)
    ax.axhline(y=0, color="black", linewidth=0.5)
    ax.set_xlabel("Strike Price")
    ax.set_ylabel("QAE - BS Difference")
    ax.set_title("Pricing Error by Strike")
    ax.grid(True, alpha=0.3)

    # Delta surface
    ax = axes[2]
    im = ax.imshow(
        delta_surface, aspect="auto", cmap="RdYlGn",
        extent=[strike_range[0], strike_range[-1],
                spot_range[-1], spot_range[0]],
    )
    ax.set_xlabel("Strike")
    ax.set_ylabel("Spot")
    ax.set_title("Call Delta Surface")
    fig.colorbar(im, ax=ax, label="Delta")

    fig.tight_layout()
    plt.show()
else:
    print("Skipping plot (matplotlib not available)")

## 2. Portfolio Optimization: QAOA vs Classical

The Quantum Approximate Optimization Algorithm (QAOA) solves the Markowitz portfolio selection
problem by encoding it as a QUBO (Quadratic Unconstrained Binary Optimization) and mapping
it to an Ising Hamiltonian.

We compare QAOA results against a classical brute-force solver and visualize the efficient frontier.

In [ ]:
from nqpu.finance import (
    PortfolioOptimizer,
    compute_efficient_frontier,
    classical_portfolio_optimize,
)

asset_names = ["Tech", "Healthcare", "Energy", "Finance", "Consumer"]
expected_returns = np.array([0.12, 0.08, 0.10, 0.07, 0.09])
covariance = np.array([
    [0.04,  0.006, 0.002, 0.005, 0.003],
    [0.006, 0.025, 0.004, 0.003, 0.002],
    [0.002, 0.004, 0.035, 0.002, 0.005],
    [0.005, 0.003, 0.002, 0.02,  0.004],
    [0.003, 0.002, 0.005, 0.004, 0.022],
])

print(f"Assets: {asset_names}")
print(f"Expected returns: {expected_returns}")
print(f"\nCovariance matrix:")
print(covariance)

In [ ]:
# QAOA optimization (select 3 out of 5 assets)
optimizer = PortfolioOptimizer(
    num_layers=2, risk_aversion=1.0, budget=3,
)
t0 = time.time()
qaoa_result = optimizer.optimize(expected_returns, covariance)
qaoa_time = time.time() - t0

print(f"QAOA Results (budget=3, 2 layers):")
print(f"  Expected return: {qaoa_result.expected_return:.4f}")
print(f"  Variance:        {qaoa_result.variance:.6f}")
print(f"  Objective:       {qaoa_result.objective:.6f}")
print(f"  Iterations:      {qaoa_result.iterations}")
print(f"  Time:            {qaoa_time:.3f}s")
print(f"  Selected:        {[n for n, b in zip(asset_names, qaoa_result.best_bitstring) if b]}")
print(f"\n  Weights:")
for name, w in zip(asset_names, qaoa_result.weights):
    print(f"    {name:>12}: {w:.4f}")

In [ ]:
# Classical brute-force comparison
t0 = time.time()
classical = classical_portfolio_optimize(
    expected_returns, covariance, risk_aversion=1.0, budget=3,
)
classical_time = time.time() - t0

# Efficient frontier
frontier = compute_efficient_frontier(expected_returns, covariance, num_points=20)

print(f"Classical Results:")
print(f"  Expected return: {classical.expected_return:.4f}")
print(f"  Variance:        {classical.variance:.6f}")
print(f"  Objective:       {classical.objective:.6f}")
print(f"  Selected:        {[n for n, b in zip(asset_names, classical.best_bitstring) if b]}")
print(f"\nComparison:")
print(f"  {'':>14} {'QAOA':>12} {'Classical':>12}")
print(f"  {'Exp. Return':>14} {qaoa_result.expected_return:>12.4f} {classical.expected_return:>12.4f}")
print(f"  {'Variance':>14} {qaoa_result.variance:>12.6f} {classical.variance:>12.6f}")
print(f"  {'Objective':>14} {qaoa_result.objective:>12.6f} {classical.objective:>12.6f}")
print(f"  {'Time (s)':>14} {qaoa_time:>12.3f} {classical_time:>12.3f}")

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Efficient frontier
    ax = axes[0]
    f_std = [np.sqrt(pt.variance) for pt in frontier]
    f_ret = [pt.target_return for pt in frontier]
    ax.plot(f_std, f_ret, "b-", linewidth=2, label="Efficient Frontier")

    q_std = np.sqrt(qaoa_result.variance)
    c_std = np.sqrt(classical.variance)
    ax.plot(q_std, qaoa_result.expected_return, "r*",
            markersize=15, label=f"QAOA (obj={qaoa_result.objective:.4f})")
    ax.plot(c_std, classical.expected_return, "g^",
            markersize=12, label=f"Classical (obj={classical.objective:.4f})")

    ax.set_xlabel("Portfolio Std Dev")
    ax.set_ylabel("Expected Return")
    ax.set_title("Efficient Frontier with Optimal Portfolios")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Weight comparison
    ax = axes[1]
    x = np.arange(len(asset_names))
    width = 0.35
    ax.bar(x - width / 2, qaoa_result.weights, width,
           label="QAOA", color="#2196F3", alpha=0.8)
    ax.bar(x + width / 2, classical.weights, width,
           label="Classical", color="#FF5722", alpha=0.8)
    ax.set_xlabel("Asset")
    ax.set_ylabel("Weight")
    ax.set_title("Portfolio Weight Comparison")
    ax.set_xticks(x)
    ax.set_xticklabels(asset_names, rotation=45, ha="right")
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

    fig.tight_layout()
    plt.show()
else:
    print("Skipping plot (matplotlib not available)")

## 3. Risk Analysis: Quantum VaR/CVaR

Value at Risk (VaR) and Conditional VaR (CVaR / Expected Shortfall) measure portfolio
downside risk. Quantum amplitude estimation provides a quadratic speedup for estimating
tail probabilities compared to classical Monte Carlo.

We compare classical and quantum risk estimates under both Normal and Student-t return distributions.

In [ ]:
from nqpu.finance import (
    RiskAnalyzer,
    RiskConfig,
    DistributionModel,
    compute_var,
    compute_cvar,
    generate_scenarios,
    quantum_var,
    quantum_cvar,
)

# Portfolio definition (reuses the same assets)
weights = np.array([0.30, 0.20, 0.25, 0.15, 0.10])
portfolio_value = 1_000_000

# Normal distribution config
config_normal = RiskConfig(
    confidence_level=0.95,
    num_scenarios=10000,
    distribution=DistributionModel.NORMAL,
    num_qubits=4,
    seed=42,
)

# Student-t config
config_t = RiskConfig(
    confidence_level=0.95,
    num_scenarios=10000,
    distribution=DistributionModel.STUDENT_T,
    df=5.0,
    num_qubits=4,
    seed=42,
)

print(f"Portfolio: ${portfolio_value:,.0f}")
print(f"Weights: {dict(zip(asset_names, weights))}")
print(f"Confidence level: 95%")

In [ ]:
# Classical risk metrics (Normal)
analyzer_n = RiskAnalyzer(config_normal)
classical_n = analyzer_n.classical_analyze(expected_returns, covariance, weights)

print(f"Classical Risk Metrics (Normal distribution):")
print(f"  VaR (95%):     {classical_n.var:.6f}  (${classical_n.var * portfolio_value:,.0f})")
print(f"  CVaR (95%):    {classical_n.cvar:.6f}  (${classical_n.cvar * portfolio_value:,.0f})")
print(f"  Max Drawdown:  {classical_n.max_drawdown:.6f}")
print(f"  Sharpe:        {classical_n.sharpe_ratio:.4f}")
print(f"  Sortino:       {classical_n.sortino_ratio:.4f}")

In [ ]:
# Quantum-enhanced risk metrics (Normal)
quantum_n = analyzer_n.analyze(expected_returns, covariance, weights)

print(f"Quantum-Enhanced Risk Metrics (Normal):")
print(f"  VaR (95%):     {quantum_n.var:.6f}  (${quantum_n.var * portfolio_value:,.0f})")
print(f"  CVaR (95%):    {quantum_n.cvar:.6f}  (${quantum_n.cvar * portfolio_value:,.0f})")
print(f"  Method:        {quantum_n.method}")
print(f"  VaR CI:        [{quantum_n.var_confidence_interval[0]:.6f}, {quantum_n.var_confidence_interval[1]:.6f}]")

In [ ]:
# Student-t comparison (heavier tails)
analyzer_t = RiskAnalyzer(config_t)
classical_t = analyzer_t.classical_analyze(expected_returns, covariance, weights)
quantum_t = analyzer_t.analyze(expected_returns, covariance, weights)

print(f"{'Metric':>18} {'Normal-CL':>12} {'Normal-QAE':>12} {'t(5)-CL':>12} {'t(5)-QAE':>12}")
print(f"{'-'*18} {'-'*12} {'-'*12} {'-'*12} {'-'*12}")
for label, ncl, nq, tcl, tq in [
    ("VaR", classical_n.var, quantum_n.var, classical_t.var, quantum_t.var),
    ("CVaR", classical_n.cvar, quantum_n.cvar, classical_t.cvar, quantum_t.cvar),
    ("Sharpe", classical_n.sharpe_ratio, quantum_n.sharpe_ratio, classical_t.sharpe_ratio, quantum_t.sharpe_ratio),
    ("Sortino", classical_n.sortino_ratio, quantum_n.sortino_ratio, classical_t.sortino_ratio, quantum_t.sortino_ratio),
    ("Max DD", classical_n.max_drawdown, quantum_n.max_drawdown, classical_t.max_drawdown, quantum_t.max_drawdown),
]:
    print(f"{label:>18} {ncl:>12.6f} {nq:>12.6f} {tcl:>12.6f} {tq:>12.6f}")

print(f"\nStudent-t produces larger tail risk (higher VaR/CVaR) as expected.")

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # VaR/CVaR bar comparison
    ax = axes[0]
    labels = ["VaR\n(Normal)", "CVaR\n(Normal)", "VaR\n(Student-t)", "CVaR\n(Student-t)"]
    classical_vals = [
        classical_n.var * portfolio_value,
        classical_n.cvar * portfolio_value,
        classical_t.var * portfolio_value,
        classical_t.cvar * portfolio_value,
    ]
    quantum_vals = [
        quantum_n.var * portfolio_value,
        quantum_n.cvar * portfolio_value,
        quantum_t.var * portfolio_value,
        quantum_t.cvar * portfolio_value,
    ]

    x = np.arange(len(labels))
    width = 0.35
    ax.bar(x - width / 2, classical_vals, width,
           label="Classical MC", color="#2196F3", alpha=0.8)
    ax.bar(x + width / 2, quantum_vals, width,
           label="QAE-Enhanced", color="#FF5722", alpha=0.8)
    ax.set_ylabel("Dollar Risk ($)")
    ax.set_title("Risk Metrics: Classical vs Quantum")
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

    # Ratio comparison
    ax = axes[1]
    metric_labels = ["Sharpe", "Sortino"]
    x = np.arange(len(metric_labels))
    w = 0.2
    ax.bar(x - 1.5 * w, [classical_n.sharpe_ratio, classical_n.sortino_ratio], w,
           label="Normal CL", color="#2196F3", alpha=0.7)
    ax.bar(x - 0.5 * w, [quantum_n.sharpe_ratio, quantum_n.sortino_ratio], w,
           label="Normal QAE", color="#03A9F4", alpha=0.7)
    ax.bar(x + 0.5 * w, [classical_t.sharpe_ratio, classical_t.sortino_ratio], w,
           label="Student-t CL", color="#FF5722", alpha=0.7)
    ax.bar(x + 1.5 * w, [quantum_t.sharpe_ratio, quantum_t.sortino_ratio], w,
           label="Student-t QAE", color="#FF9800", alpha=0.7)
    ax.set_ylabel("Ratio")
    ax.set_title("Risk-Adjusted Return Ratios")
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis="y")

    fig.tight_layout()
    plt.show()
else:
    print("Skipping plot (matplotlib not available)")

## 4. Trading: Regime Detection & Backtesting

Quantum-inspired trading tools encode market data into quantum states for:

- **Regime Detection** -- Classify market conditions (bull, bear, sideways, volatile) via quantum state fidelity
- **Signal Generation** -- Extract buy/sell/hold signals from quantum measurements
- **Momentum** -- Quantum walk dynamics for directional momentum detection
- **Backtesting** -- Walk-forward evaluation with transaction costs and position sizing

In [ ]:
from nqpu.trading import (
    QuantumRegimeDetector,
    QuantumSignalGenerator,
    QuantumBacktester,
    QuantumMomentum,
    MarketRegime,
    PerformanceMetrics,
)

# Generate synthetic price data (Geometric Brownian Motion)
np.random.seed(42)
n_days = 500
dt = 1 / 252
mu, sigma = 0.08, 0.25
daily_returns = np.random.normal(mu * dt, sigma * np.sqrt(dt), n_days)
prices = 100.0 * np.cumprod(1 + daily_returns)

print(f"Synthetic data: {n_days} days, drift={mu}, vol={sigma}")
print(f"Price range: [{prices.min():.2f}, {prices.max():.2f}]")
print(f"Final price: {prices[-1]:.2f}")

In [ ]:
# Regime detection
log_returns = np.diff(np.log(np.maximum(prices, 1e-12)))
detector = QuantumRegimeDetector(n_qubits=4)
detector.fit(log_returns, window=20)
regimes = detector.detect_series(log_returns, window=20)

regime_counts = {}
for r in regimes:
    name = str(r)
    regime_counts[name] = regime_counts.get(name, 0) + 1

print(f"Regime Distribution ({len(regimes)} windows):")
for regime, count in sorted(regime_counts.items()):
    pct = 100.0 * count / len(regimes)
    bar = '#' * int(pct / 2)
    print(f"  {regime:>10}: {count:>4} ({pct:5.1f}%) {bar}")

In [ ]:
# Signal generation and momentum
sig_gen = QuantumSignalGenerator(n_qubits=4, seed=42)
signals = sig_gen.generate(prices, window=20)

buy_count = sum(1 for s in signals if s.label == "buy")
sell_count = sum(1 for s in signals if s.label == "sell")
hold_count = sum(1 for s in signals if s.label == "hold")

print(f"Signal Statistics ({len(signals)} signals):")
print(f"  Buy:  {buy_count:>4} ({100*buy_count/len(signals):.1f}%)")
print(f"  Sell: {sell_count:>4} ({100*sell_count/len(signals):.1f}%)")
print(f"  Hold: {hold_count:>4} ({100*hold_count/len(signals):.1f}%)")
print(f"  Avg confidence: {np.mean([s.confidence for s in signals]):.3f}")

# Momentum
momentum = QuantumMomentum(n_levels=16, n_steps=5)
mom_values = momentum.compute(prices, window=20)
print(f"\nQuantum Momentum ({len(mom_values)} values):")
print(f"  Mean:  {mom_values.mean():>+.4f}")
print(f"  Std:   {mom_values.std():>.4f}")
print(f"  Range: [{mom_values.min():.4f}, {mom_values.max():.4f}]")

In [ ]:
# Backtesting
backtester = QuantumBacktester(
    transaction_cost_bps=10.0, max_position=1.0, seed=42,
)
bt_result = backtester.run(prices, window=20)
perf = bt_result["metrics"]

print("Backtest Performance:")
print(perf.summary())

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # Price with regime coloring
    ax = axes[0, 0]
    regime_colors = {
        "bull": "#4CAF50", "bear": "#F44336",
        "sideways": "#FFC107", "volatile": "#9C27B0",
    }
    ax.plot(prices, color="black", linewidth=0.8, alpha=0.7)
    offset = len(prices) - len(regimes)
    for i in range(len(regimes) - 1):
        r_name = str(regimes[i])
        color = regime_colors.get(r_name, "#999999")
        ax.axvspan(i + offset, i + offset + 1, alpha=0.15, color=color)
    ax.set_xlabel("Day")
    ax.set_ylabel("Price")
    ax.set_title("Price Series with Quantum Regime Detection")
    legend_patches = [
        Patch(facecolor=c, alpha=0.4, label=n.capitalize())
        for n, c in regime_colors.items()
    ]
    ax.legend(handles=legend_patches, loc="upper left", fontsize=8)

    # Equity curve
    ax = axes[0, 1]
    equity = bt_result["equity_curve"]
    ax.plot(equity, color="#2196F3", linewidth=1.5)
    ax.axhline(y=1.0, color="gray", linestyle="--", linewidth=0.5)
    ax.fill_between(range(len(equity)), 1.0, equity,
                    where=equity >= 1.0, alpha=0.2, color="#4CAF50")
    ax.fill_between(range(len(equity)), 1.0, equity,
                    where=equity < 1.0, alpha=0.2, color="#F44336")
    ax.set_xlabel("Trading Day")
    ax.set_ylabel("Equity (normalized)")
    ax.set_title(f"Equity Curve (Sharpe: {perf.sharpe_ratio:.2f})")
    ax.grid(True, alpha=0.3)

    # Momentum
    ax = axes[1, 0]
    ax.plot(mom_values, color="#9C27B0", linewidth=0.8, alpha=0.8)
    ax.axhline(y=0, color="gray", linewidth=0.5)
    ax.fill_between(range(len(mom_values)), 0, mom_values,
                    where=mom_values > 0, alpha=0.3, color="#4CAF50")
    ax.fill_between(range(len(mom_values)), 0, mom_values,
                    where=mom_values < 0, alpha=0.3, color="#F44336")
    ax.set_xlabel("Window")
    ax.set_ylabel("Momentum")
    ax.set_title("Quantum Walk Momentum")
    ax.grid(True, alpha=0.3)

    # Signal distribution
    ax = axes[1, 1]
    directions = [s.direction for s in signals]
    confidences = [s.confidence for s in signals]
    colors = ["#4CAF50" if s.label == "buy"
              else "#F44336" if s.label == "sell"
              else "#FFC107" for s in signals]
    ax.scatter(directions, confidences, c=colors, alpha=0.4, s=10)
    ax.set_xlabel("Signal Direction")
    ax.set_ylabel("Confidence")
    ax.set_title("Signal Distribution (Green=Buy, Red=Sell, Yellow=Hold)")
    ax.axvline(x=0.15, color="gray", linestyle="--", linewidth=0.5)
    ax.axvline(x=-0.15, color="gray", linestyle="--", linewidth=0.5)
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
else:
    print("Skipping plot (matplotlib not available)")

## Summary

This notebook demonstrated four pillars of quantum finance using the nQPU SDK:

**Key Takeaways:**

1. **Option Pricing** -- QAE provides option prices that converge toward Black-Scholes analytical
   values, with the advantage of naturally extending to exotic options (Asian, Barrier) where
   no closed-form solution exists. On real quantum hardware, QAE achieves a quadratic speedup
   over classical Monte Carlo.

2. **Portfolio Optimization** -- QAOA finds near-optimal portfolio allocations for the Markowitz
   mean-variance problem. For small instances (5 assets), classical brute-force is faster, but
   QAOA scales polynomially while brute-force scales exponentially -- the crossover point is
   around 20-30 assets.

3. **Risk Analysis** -- Quantum VaR/CVaR estimation refines classical Monte Carlo estimates.
   The Student-t distribution produces larger tail risk estimates than Normal, reflecting
   real-world fat tails. Both classical and quantum methods agree on risk ordering.

4. **Trading Signals** -- Quantum state encoding reveals market regimes and generates trading
   signals that can be backtested with realistic transaction costs. Quantum walk momentum
   captures directional bias through ballistic spreading rather than diffusive random walks.

**Next Steps:**
- Scale to larger portfolios (10-50 assets) using noise-aware QAOA
- Integrate with real market data feeds
- Run on actual quantum hardware via nQPU backend connectors
- Combine regime detection with risk management for adaptive position sizing